In [0]:
# =============================================================================
# 04_gold: cria as tabelas analíticas a partir da Silver
# Entrada: workspace.silver.sgs_series
# Saídas:  workspace.gold.dolar_mensal e workspace.gold.juro_real_mensal
# =============================================================================

from pyspark.sql import Window
from pyspark.sql import functions as F

CATALOG = "workspace"
silver = spark.table(f"{CATALOG}.silver.sgs_series")


# Função auxiliar: transforma qualquer data no 1º dia do mês dela.
# Ex.: 2026-11-17 -> 2026-11-01. É assim que agrupamos "por mês".
# date_trunc devolve timestamp; o cast("date") tira a parte da hora.
def inicio_do_mes(coluna):
    return F.date_trunc("MONTH", coluna).cast("date")


# -----------------------------------------------------------------------------
# TABELA 1: DÓLAR MENSAL
# De ~22 cotações diárias por mês para 1 linha de resumo por mês.
# -----------------------------------------------------------------------------
dolar_mensal = (
    silver.filter(F.col("serie_codigo") == 1)                  # só a série do dólar
    .groupBy(inicio_do_mes(F.col("data_referencia")).alias("mes"))  # agrupa por mês
    .agg(
        F.round(F.avg("valor"), 4).alias("cotacao_media"),    # média do mês
        F.min("valor").alias("cotacao_minima"),               # menor cotação
        F.max("valor").alias("cotacao_maxima"),               # maior cotação
        # max_by(valor, data) = "o valor da linha com a MAIOR data"
        # ou seja, a cotação do último dia útil do mês (fechamento)
        F.max_by("valor", "data_referencia").alias("cotacao_fechamento"),
        F.count("*").alias("dias_com_cotacao"),              # dias úteis com dado
    )
    # Variação % contra o mês anterior.
    # lag() pega o valor da linha ANTERIOR na ordem definida pela janela
    # (aqui, ordenada por mês). No primeiro mês não há anterior -> NULL.
    # Fórmula: (fechamento_atual / fechamento_anterior - 1) * 100
    .withColumn(
        "variacao_mensal_pct",
        F.round(
            (F.col("cotacao_fechamento") / F.lag("cotacao_fechamento").over(Window.orderBy("mes")) - 1) * 100,
            2,
        ),
    )
    .withColumn("mes_completo", F.col("mes") < inicio_do_mes(F.current_date()))
)

# -----------------------------------------------------------------------------
# TABELA 2: JURO REAL MENSAL
# Juro real = quanto a Selic rende ACIMA da inflação.
# Precisamos de duas séries no mesmo nível (mensal) para cruzar.
# -----------------------------------------------------------------------------

# Selic: a série é diária, então pegamos o valor do último dia de cada mês
selic = (
    silver.filter(F.col("serie_codigo") == 432)
    .groupBy(inicio_do_mes(F.col("data_referencia")).alias("mes"))
    .agg(F.max_by("valor", "data_referencia").alias("selic_meta_aa"))
)

# Janela "móvel" de 12 linhas: a linha atual + as 11 anteriores.
# Como o IPCA tem 1 linha por mês, isso equivale aos últimos 12 meses.
janela_12m = Window.orderBy("mes").rowsBetween(-11, 0)

ipca = (
    silver.filter(F.col("serie_codigo") == 433)
    .select(inicio_do_mes(F.col("data_referencia")).alias("mes"), F.col("valor").alias("ipca_mes"))
    # IPCA acumulado em 12 meses, COMPOSTO (inflação sobre inflação):
    #   (1 + t1) * (1 + t2) * ... * (1 + t12) - 1
    # O Spark não tem "produto" em janela, então usamos um truque matemático:
    #   produto(x) = exp(soma(log(x)))
    # log1p(t) = log(1 + t). Dividimos por 100 porque 0,5% = 0.005.
    .withColumn("ipca_12m", (F.exp(F.sum(F.log1p(F.col("ipca_mes") / 100)).over(janela_12m)) - 1) * 100)
    # Quantos meses entraram na janela. Nos 11 primeiros meses da série,
    # a janela está incompleta e o acumulado seria enganoso.
    .withColumn("meses_na_janela", F.count("*").over(janela_12m))
)

juro_real_mensal = (
    selic.join(ipca, "mes")                   # junta Selic e IPCA pelo mês
    .filter(F.col("meses_na_janela") == 12)  # só meses com 12 meses completos de IPCA
    .select(
        "mes",
        "selic_meta_aa",
        "ipca_mes",
        F.round("ipca_12m", 2).alias("ipca_12m"),
        # Fórmula de Fisher: (1 + juro nominal) / (1 + inflação) - 1
        # Mais correta que "Selic - IPCA", porque taxas se multiplicam.
        F.round(((1 + F.col("selic_meta_aa") / 100) / (1 + F.col("ipca_12m") / 100) - 1) * 100, 2).alias("juro_real_aa"),
    )
)

# -----------------------------------------------------------------------------
# GRAVAÇÃO
# overwrite: a Gold é recalculada inteira a cada execução (são poucas linhas).
# overwriteSchema: permite mudar colunas se você alterar o cálculo depois.
# -----------------------------------------------------------------------------
for nome, df in [("dolar_mensal", dolar_mensal), ("juro_real_mensal", juro_real_mensal)]:
    (
        df.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.gold.{nome}")
    )

# Conferência: os 12 meses mais recentes do juro real
display(spark.table(f"{CATALOG}.gold.juro_real_mensal").orderBy(F.col("mes").desc()).limit(12))

Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.table("workspace.gold.dolar_mensal").orderBy("mes", ascending=False))